# Agentic Financial Dispute Engine - Evaluation

This notebook demonstrates a simple evaluation framework for our Agentic RAG system. We will evaluate:
1. **Retrieval Relevance**: Did the FAISS index return chunks relevant to the query?
2. **Groundedness**: Is the generated response supported by the retrieved context?
3. **Response Quality**: Does the final report address the user's prompt?

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

from src.rag_pipeline import rag_pipeline
from src.graph import app_graph
import pandas as pd

## 1. Evaluating Retrieval Relevance

In [ ]:
test_queries = [
    ("What is the time limit for a chargeback?", "chargeback_policy.pdf"),
    ("Are digital goods refundable?", "refund_and_return_policy.pdf"),
]

def evaluate_retrieval(queries):
    correct = 0
    for query, expected_source in queries:
        results = rag_pipeline.retrieve(query, k=3)
        sources = [doc.metadata.get('source', '') for doc in results]
        
        if any(expected_source in source for source in sources):
            correct += 1
            print(f"✅ PASS: '{query}' -> Found in {expected_source}")
        else:
            print(f"❌ FAIL: '{query}' -> Expected {expected_source}, got {sources}")
            
    print(f"\nRetrieval Accuracy: {correct}/{len(queries)} ({(correct/len(queries))*100}%)")

evaluate_retrieval(test_queries)

## 2. End-to-End Evaluation (Groundedness & Quality)
We run a full graph execution and manually inspect (or use an LLM-as-a-judge) to verify the report.

In [ ]:
# Load a sample transaction to test
try:
    df = pd.read_csv('../data/structured/transactions.csv')
    sample_txn = df['transaction_id'].iloc[0]
    query = f"Investigate transaction {sample_txn}. Customer says it was fraud."
    
    print(f"Running query: {query}\n")
    result = app_graph.invoke({"query": query})
    
    print("=== FINAL REPORT ===")
    print(result['final_report'])
    print(f"\nConfidence Score: {result['confidence_score']}")
    
except Exception as e:
    print(f"Make sure to run data generation first and have Ollama running: {e}")